# Week 4: pixel-level rescore of Week 3 models

Re-score Week 3 OOF predictions against the fraction-only GT
([`datasets/week4_pixel_gt`](../../datasets/week4_pixel_gt)): `frac_res`, `frac_nres`, `frac_empty`.

Primary metric: mean MAE across the three fractions (lower is better). Protocol: join on `pixel_id`, then pooled OOF and leave-one-region-out means.

Rebuild metrics: `.venv/bin/python scripts/eval_week4_pixel.py --run-id week3_rescore_v1`


## Models scored

| Key | Week 3 artifact | How fractions are formed |
|---|---|---|
| `pixel_3class_reg` | `outputs/week3_pixel_3class/oof_predictions.parquet` | Soft-fraction MLP head (`pred_frac_*`) |
| `neighborhood_k3_reg` | `outputs/week3_pixel_neighborhood_k3/` | 3×3 CNN soft-fraction head |
| `pixel_ae` | `outputs/week3_pixel_ae/` | `p_res` → `(p_res, 1−p_res, 0)` (no empty mass) |
| `fraction_regression` | `outputs/week3_fraction_regression/` | Ridge `(pred_res, pred_nres)` + derived empty |

Binary models never predict empty, so their `frac_empty` MAE equals the GT empty mean (~0.37) on joined cells.


In [1]:
from pathlib import Path
import json
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = HERE
for p in [HERE, *HERE.parents]:
    if (p / "outputs" / "week4_pixel_eval" / "week3_rescore_v1" / "summary.json").exists():
        ROOT = p
        break

RUN = ROOT / "outputs" / "week4_pixel_eval" / "week3_rescore_v1"
summary = pd.read_csv(RUN / "summary_metrics.csv")
per_region = pd.read_csv(RUN / "per_region_metrics.csv")
manifest = json.loads((RUN / "summary.json").read_text())

cols = [
    "model", "n_joined",
    "oof_mae_frac_res", "oof_mae_frac_nres", "oof_mae_frac_empty",
    "oof_mae_mean", "loro_mae_mean_mean",
]
board = summary[cols].sort_values("oof_mae_mean").reset_index(drop=True)
print(f"run_id={manifest['run_id']}  n_gt={manifest['n_gt']:,}")
display(board.round(4))


run_id=week3_rescore_v1  n_gt=283,256


,model,n_joined,oof_mae_frac_res,oof_mae_frac_nres,oof_mae_frac_empty,oof_mae_mean,loro_mae_mean_mean
0,pixel_3class_reg,283119,0.1685,0.3676,0.3206,0.2856,0.2844
1,neighborhood_k3_reg,279708,0.0574,0.3937,0.4058,0.2856,0.2844
2,pixel_ae,283119,0.3842,0.3372,0.3721,0.3645,0.3657
3,fraction_regression,283119,0.3913,0.3342,0.3721,0.3658,0.3665


## Leaderboard (week4 GT)

Soft-fraction heads that model empty beat binary AE / Ridge on mean MAE (~0.286 vs ~0.365).
`pixel_3class_reg` and `neighborhood_k3_reg` nearly tie on the mean, but for different reasons (see per-fraction bars).

![MAE ladder](../../outputs/week4_pixel_eval/week3_rescore_v1/figures/mae_ladder.png)

![Per-fraction MAE](../../outputs/week4_pixel_eval/week3_rescore_v1/figures/mae_by_fraction.png)


## Takeaways

1. Empty matters for fraction eval. Binary Week 3 models (`pixel_ae`, Ridge) put zero mass on empty, so empty MAE ≈ GT mean empty (0.37) and mean MAE rises to ~0.365.
2. Best mean MAE is `pixel_3class_reg` (0.286), most even across the three fracs (res 0.168, nres 0.368, empty 0.321).
3. Neighborhood CNN is best on `frac_res` (MAE 0.057) but worse on empty (0.406); mean MAE ties 3-class by coincidence.
4. Week 3 Ridge MAE on `frac_res` (0.391) matches this rescore exactly: week4 coverage fractions for overlapping pixels use the same construction as week3.
5. Join coverage: 283,119 / 283,256 for AE-grid models; neighborhood drops a few thousand pixels without `pixel_id` overlap.


In [2]:
frac_cols = ["oof_mae_frac_res", "oof_mae_frac_nres", "oof_mae_frac_empty", "oof_mae_mean"]
print("Best model per fraction (lowest OOF MAE):")
for c in frac_cols:
    i = summary[c].idxmin()
    print(f"  {c}: {summary.loc[i, 'model']} = {summary.loc[i, c]:.4f}")


Best model per fraction (lowest OOF MAE):
  oof_mae_frac_res: neighborhood_k3_reg = 0.0574
  oof_mae_frac_nres: fraction_regression = 0.3342
  oof_mae_frac_empty: pixel_3class_reg = 0.3206
  oof_mae_mean: pixel_3class_reg = 0.2856


## LORO by metro

![Mean MAE by region](../../outputs/week4_pixel_eval/week3_rescore_v1/figures/mae_by_region.png)


In [3]:
pivot = (
    per_region.pivot(index="region", columns="model", values="mae_mean")
    .sort_index()
)
display(pivot.round(3))
wins = pivot.idxmin(axis=1).value_counts()
print("Metros won (lowest mean MAE):")
display(wins)


model,fraction_regression,neighborhood_k3_reg,pixel_3class_reg,pixel_ae
region,,,,
Atlanta_SE,0.365,0.247,0.274,0.366
Boston_NE,0.404,0.284,0.302,0.400
Chicago_MW,0.303,0.219,0.266,0.286
Denver_W,0.394,0.380,0.312,0.399
Houston_S,0.415,0.357,0.314,0.421
LA_W,0.406,0.262,0.289,0.400
NYC_NE,0.359,0.280,0.297,0.358
Phoenix_SW,0.290,0.216,0.222,0.287
SF_W,0.344,0.228,0.279,0.343


Metros won (lowest mean MAE):


neighborhood_k3_reg    7
pixel_3class_reg       3
Name: count, dtype: int64

## Artifacts

| Path | Contents |
|---|---|
| `outputs/week4_pixel_eval/week3_rescore_v1/summary_metrics.csv` | Pooled + LORO metrics per model |
| `outputs/week4_pixel_eval/week3_rescore_v1/per_region_metrics.csv` | Per-metro MAE/RMSE/R² |
| `outputs/week4_pixel_eval/week3_rescore_v1/summary.json` | Manifest + leaderboard |
| `outputs/week4_pixel_eval/week3_rescore_v1/figures/` | Ladder / per-fraction / by-region plots |
| `scripts/eval_week4_pixel.py` | Reusable scorer |
